# Supplementary Note 1 - The CORDEX2Energy conversion pipeline

Figures S1–S16 and the inset panels of main Figure 1: the chains and their warming-level windows, the bias-corrected climate, the demand model and each conversion step.

In [ ]:
from cordex2energy_analysis.config import SEASON_ORDER, setup
from cordex2energy_analysis.demand_validation import (by_iso2, cooling_performance, curvature_candidates,
                                                      heating_performance, load_calibration_data,
                                                      production_curvatures)
from cordex2energy_analysis.figures_pipeline import (climate_figure, conversion_figure, figure_s1,
                                                     figure_s2, figure_s8, figure_s9, figure_s10,
                                                     pipeline_panels)
from cordex2energy_analysis.pipeline_inputs import (annual_country_cooling, chain_table, conversion_rows,
                                                    demand_fields, driving_gcm_rows, european_seasonal_change,
                                                    generation_fields, load_cmip5_gmst, load_gwl_grids)
from cordex2energy_analysis.plotting_publication import publication_rc, save

context = setup("si_note1_pipeline")

In [ ]:
climate_annual = load_gwl_grids(context.paths.gwl_grids, "climate", context.period_names)
climate_seasonal = load_gwl_grids(context.paths.gwl_grids, "climate", context.period_names, seasonal=True)
energy_annual = load_gwl_grids(context.paths.gwl_grids, "", context.period_names)
energy_seasonal = load_gwl_grids(context.paths.gwl_grids, "", context.period_names, seasonal=True)
spain_cooling = annual_country_cooling(context, "Spain")
gmst = load_cmip5_gmst(context.paths.processed)
seasonal_change = european_seasonal_change(context)
chain_table(context.models, context.period_windows)

In [ ]:
calibration = load_calibration_data(context.paths.processed / "energy" / "inputs")
heating_scores, _ = heating_performance(calibration)
cooling_scores = cooling_performance(calibration)
cooling_candidates = curvature_candidates(calibration)

In [ ]:
generation_annual, generation_seasonal = generation_fields(context)
demand_annual, demand_seasonal = demand_fields(context)
country_sources = ({**demand_annual, **generation_annual}, {**demand_seasonal, **generation_seasonal},
                   energy_annual, energy_seasonal)

## Figure 1 insets - Annual cooling demand in Spain
**In:** hourly cooling demand of the nine chains. **Shows:** annual cooling demand per chain and its ensemble median over each 20-year warming-level window, one panel per level on a shared y scale.

In [ ]:
with publication_rc():
    for suffix, fig in pipeline_panels(spain_cooling, context.period_names).items():
        save(fig, f"pipeline_cooling_spain_{suffix}", context.figure_dir / "pipeline", max_width_mm=60)

## S1 - The driving GCMs and the warming-level windows
**In:** CMIP5 RCP8.5 GMST anomalies and the warming-level windows. **Shows:** the five driving GCMs within the CMIP5 ensemble (a) and the 20-year window each defines at every level (b).

In [ ]:
with publication_rc():
    save(figure_s1(gmst, driving_gcm_rows(context.models), context.period_windows, context.style),
         "S1_driving_gcms_and_windows", context.figure_dir)

## S2 - European precipitation and runoff against warming, by season
**In:** seasonal gridded climate means at +1.5 °C and +3.0 °C, CERRA and land masks. **Shows:** each chain's change in precipitation (a) and runoff (b) against its temperature change, per season.

In [ ]:
with publication_rc():
    save(figure_s2(seasonal_change), "S2_european_t_p_runoff_seasonal", context.figure_dir)

## S3 - The bias-corrected climate variables, annual
**In:** annual gridded climate means at the three warming levels. **Shows:** tas, rsds, sfcWind, pr and mrro at +1.5 °C and their median change at +2.0 °C and +3.0 °C, hatched where chains disagree on the sign.

In [ ]:
with publication_rc():
    save(climate_figure(climate_annual, context.countries_print, context.style, height_mm=174),
         "S3_climate_annual", context.figure_dir)

## S4–S7 - The bias-corrected climate variables, by season
**In:** seasonal gridded climate means at the three warming levels. **Shows:** the S3 layout for DJF (S4), MAM (S5), JJA (S6) and SON (S7).

In [ ]:
season_stems = {"DJF": "S4_climate_DJF", "MAM": "S5_climate_MAM", "JJA": "S6_climate_JJA",
                "SON": "S7_climate_SON"}
with publication_rc():
    for season in SEASON_ORDER:
        save(climate_figure(climate_seasonal, context.countries_print, context.style, season=season,
                            height_mm=186),
             season_stems[season], context.figure_dir)

## S8 - Runoff and the routed channel discharge
**In:** annual gridded mrro and Qrouted at +1.5 °C. **Shows:** the ensemble-median runoff (a, linear) beside the discharge it is routed into (b, logarithmic).

In [ ]:
with publication_rc():
    save(figure_s8(climate_annual[context.reference], context.countries_print), "S8_runoff_routing",
         context.figure_dir)

## S9 - Demand model performance
**In:** the production demand coefficients, JRC-IDEES national energy and CERRA degree days, 2000–2020. **Shows:** per-country NRMSE (a, b) and interannual correlation (c, d) of modelled heating and cooling energy.

In [ ]:
with publication_rc():
    save(figure_s9(by_iso2(heating_scores, cooling_scores, "nrmse_pct"),
                   by_iso2(heating_scores, cooling_scores, "r"), context.countries),
         "S9_demand_model_performance", context.figure_dir)

## S10 - The fitted cooling adoption terms
**In:** the production cooling curvatures and every candidate of the curvature selection. **Shows:** the adoption share against HRE with the observed CDD (a) and each candidate curvature with its 1 %-RSS interval (b).

In [ ]:
with publication_rc():
    save(figure_s10(production_curvatures(calibration.coefficients), cooling_candidates,
                    calibration.cdd_fit.mean()),
         "S10_cooling_adoption_terms", context.figure_dir)

## S11 - Heating demand
**In:** per-country heating demand of the nine chains. **Shows:** the annual and seasonal level at +1.5 °C and its median change at +2.0 °C and +3.0 °C.

In [ ]:
with publication_rc():
    save(conversion_figure(conversion_rows("Heating demand", *country_sources), context.countries_print,
                           context.style),
         "S11_heating_demand", context.figure_dir)

## S12 - Cooling demand
**In:** per-country cooling demand of the nine chains. **Shows:** the annual and seasonal level at +1.5 °C and its median change at +2.0 °C and +3.0 °C.

In [ ]:
with publication_rc():
    save(conversion_figure(conversion_rows("Cooling demand", *country_sources), context.countries_print,
                           context.style),
         "S12_cooling_demand", context.figure_dir)

## S13 - Solar power
**In:** gridded solar capacity factors. **Shows:** the annual and seasonal capacity factor at +1.5 °C and its median change at +2.0 °C and +3.0 °C, all rows on shared scales.

In [ ]:
with publication_rc():
    save(conversion_figure(conversion_rows("Solar power", *country_sources), context.countries_print,
                           context.style, gridded=True),
         "S13_solar_power", context.figure_dir)

## S14 - Wind power
**In:** gridded wind capacity factors. **Shows:** the annual and seasonal capacity factor at +1.5 °C and its median change at +2.0 °C and +3.0 °C, all rows on shared scales.

In [ ]:
with publication_rc():
    save(conversion_figure(conversion_rows("Wind power", *country_sources), context.countries_print,
                           context.style, gridded=True),
         "S14_wind_power", context.figure_dir)

## S15 - Run-of-river generation
**In:** per-country run-of-river generation of the paper's run. **Shows:** the annual and seasonal level at +1.5 °C and its median change at +2.0 °C and +3.0 °C.

In [ ]:
with publication_rc():
    save(conversion_figure(conversion_rows("Run-of-river generation", *country_sources),
                           context.countries_print, context.style),
         "S15_run_of_river", context.figure_dir)

## S16 - Reservoir inflow
**In:** per-country reservoir inflow of the nine chains. **Shows:** the annual and seasonal level at +1.5 °C and its median change at +2.0 °C and +3.0 °C.

In [ ]:
with publication_rc():
    save(conversion_figure(conversion_rows("Reservoir inflow", *country_sources), context.countries_print,
                           context.style),
         "S16_reservoir_inflow", context.figure_dir)